1. Import Library

In [2]:
import pandas as pd
import re
import unicodedata
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.utils import resample

2. Load Dataset

In [3]:
# Sesuaikan path dengan lokasi file kamu
path_input = '../data/processed/data_final.csv'

df = pd.read_csv(path_input)

print(f'Total baris dimuat : {len(df)}')
print(f'Kolom              : {df.columns.tolist()}')
print(f'\nDistribusi sumber:')
print(df['sumber'].value_counts())
display(df.head(3))

Total baris dimuat : 11545
Kolom              : ['sumber', 'tanggal', 'username', 'teks_bersih', 'urgensi', 'sentimen', 'lokasi_insiden', 'kategori_infrastruktur', 'kategori_lingkungan', 'kategori_air_sanitasi', 'kategori_bencana', 'kategori_transportasi', 'kategori_pelayanan_publik', 'kategori_ekonomi', 'kategori_keamanan', 'kategori_pendidikan', 'kategori_kesehatan']

Distribusi sumber:
sumber
berita_kaggle    10000
twitter           1545
Name: count, dtype: int64


,sumber,tanggal,username,teks_bersih,urgensi,sentimen,lokasi_insiden,kategori_infrastruktur,kategori_lingkungan,kategori_air_sanitasi,kategori_bencana,kategori_transportasi,kategori_pelayanan_publik,kategori_ekonomi,kategori_keamanan,kategori_pendidikan,kategori_kesehatan
0,berita_kaggle,2026-03-26,@tempodotco,baca berita dengan sedikit iklan klik di sini ...,Sedang,Negatif,jakarta,1,0,1,0,0,1,1,1,0,0
1,berita_kaggle,2026-03-26,@kompascom,dinas lingkungan hidup dlh provinsi dki jakart...,Tinggi,Netral,indonesia,0,1,1,0,0,0,0,0,0,0
2,berita_kaggle,2026-03-26,@kompascom,kompas com menteri dalam negeri mendagri muham...,Rendah,Netral,Tidak diketahui,0,0,1,0,0,0,0,0,0,0


3. Drop Kolom yang Tidak Relevan untuk Model

In [4]:
# Kolom username dan lokasi_insiden tidak dibutuhkan untuk klasifikasi
kolom_drop = ['username', 'lokasi_insiden']

df = df.drop(columns=kolom_drop)

print(f'✅ Kolom {kolom_drop} berhasil dihapus')
print(f'Kolom tersisa: {df.columns.tolist()}')

✅ Kolom ['username', 'lokasi_insiden'] berhasil dihapus
Kolom tersisa: ['sumber', 'tanggal', 'teks_bersih', 'urgensi', 'sentimen', 'kategori_infrastruktur', 'kategori_lingkungan', 'kategori_air_sanitasi', 'kategori_bencana', 'kategori_transportasi', 'kategori_pelayanan_publik', 'kategori_ekonomi', 'kategori_keamanan', 'kategori_pendidikan', 'kategori_kesehatan']


4. Cleaning & Normalisasi Data Berita Kaggle

4.1. Hapus Noise Artefak Berita (Iklan & Frasa Navigasi)

In [5]:
# Frasa-frasa iklan dan navigasi situs berita yang tidak relevan
POLA_NOISE_BERITA = [
    r'baca berita dengan sedikit iklan\s*',
    r'klik di sini\s*',
    r'scroll ke bawah untuk melanjutkan membaca\s*',
    r'pilihan editor\s+[a-z\s&,]+?(?=\s{2,}|\Z)',
    r'gambas[a-z\s:]*video[a-z0-9\s]*',
    r'lihat juga\s*',
    r'baca juga\s*',
    r'copyright[^\s]*\s*',
]

def hapus_noise_berita(teks):
    if not isinstance(teks, str):
        return teks

    # Hapus setiap pola noise satu per satu
    for pola in POLA_NOISE_BERITA:
        teks = re.sub(pola, ' ', teks, flags=re.IGNORECASE)

    # Rapikan spasi yang tersisa
    teks = re.sub(r' {2,}', ' ', teks).strip()
    return teks

mask_kaggle = df['sumber'] == 'berita_kaggle'

df.loc[mask_kaggle, 'teks_bersih'] = df.loc[mask_kaggle, 'teks_bersih'].apply(hapus_noise_berita)

print('✅ Noise iklan & navigasi berhasil dihapus dari data berita_kaggle')

✅ Noise iklan & navigasi berhasil dihapus dari data berita_kaggle


4.2. Ambil 50 Kata Pertama sebagai Konteks Inti

In [6]:
# Teks berita kaggle sangat panjang (rata-rata 2.000+ karakter)
# Ambil 50 kata pertama sebagai konteks inti yang cukup untuk klasifikasi
BATAS_KATA = 50

def ambil_n_kata(teks, n=BATAS_KATA):
    if not isinstance(teks, str):
        return teks
    kata = teks.split()
    return ' '.join(kata[:n])

panjang_sebelum = df.loc[mask_kaggle, 'teks_bersih'].str.split().str.len().mean()

df.loc[mask_kaggle, 'teks_bersih'] = df.loc[mask_kaggle, 'teks_bersih'].apply(ambil_n_kata)

panjang_sesudah = df.loc[mask_kaggle, 'teks_bersih'].str.split().str.len().mean()

print(f'✅ Rata-rata kata sebelum : {panjang_sebelum:.0f} kata')
print(f'✅ Rata-rata kata sesudah  : {panjang_sesudah:.0f} kata')
print(f'\nContoh hasil (3 baris pertama):')
display(df.loc[mask_kaggle, ['sumber', 'teks_bersih']].head(3))

✅ Rata-rata kata sebelum : 326 kata
✅ Rata-rata kata sesudah  : 50 kata

Contoh hasil (3 baris pertama):


,sumber,teks_bersih
0,berita_kaggle,tempo co jakarta gubernur jakarta pramono anun...
1,berita_kaggle,dinas lingkungan hidup dlh provinsi dki jakart...
2,berita_kaggle,kompas com menteri dalam negeri mendagri muham...


5. Cleaning & Normalisasi Data Twitter

5.1. Bersihkan Karakter Unicode & Noise Khusus Twitter

In [7]:
def bersihkan_teks_twitter(teks):
    if not isinstance(teks, str):
        return teks

    # Normalisasi NFKC: ubah huruf styled (bold/italic unicode) ke huruf biasa
    # Contoh: '𝐻eavenina' → 'Heavenina'
    teks = unicodedata.normalize('NFKC', teks)

    # Hapus karakter zero-width dan invisible yang tidak kelihatan
    teks = re.sub(r'[\u200B-\u200F\u00AD\uFEFF\u202A-\u202F\u2060]', '', teks)

    # Hapus emoji
    teks = re.sub(r'[\U0001F000-\U0001FFFF\U00002600-\U000027BF\U0000FE00-\U0000FE0F]', '', teks)

    # Hapus simbol dekorasi seperti ★, ✩, 【, 】
    teks = re.sub(r'[\u2460-\u24FF\u2500-\u257F\u25A0-\u25FF\u2700-\u27BF\u3000-\u303F]', '', teks)

    # Hapus combining diacritics yang tidak lazim
    teks = re.sub(r'[\u0300-\u036F]', '', teks)

    # Hapus aksara Devanagari (bahasa India: Hindi, dll)
    teks = re.sub(r'[\u0900-\u097F]+', '', teks)

    # Hapus semua karakter non-latin yang tersisa
    teks = re.sub(r'[^\x20-\x7E\u00C0-\u024F]', '', teks)

    # Rapikan spasi
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

mask_twitter = df['sumber'] == 'twitter'

df.loc[mask_twitter, 'teks_bersih'] = df.loc[mask_twitter, 'teks_bersih'].apply(bersihkan_teks_twitter)

print(f'✅ Cleaning karakter unicode & noise selesai untuk {mask_twitter.sum()} baris twitter')

✅ Cleaning karakter unicode & noise selesai untuk 1545 baris twitter


5.2. Lowercase, Hapus Tanda Baca, & Seragamkan Format dengan Kaggle

In [8]:
def normalisasi_twitter(teks, n=BATAS_KATA):
    if not isinstance(teks, str):
        return teks

    # Lowercase agar seragam dengan data kaggle
    teks = teks.lower()

    # Hapus tanda baca, pertahankan huruf, angka, dan spasi
    teks = re.sub(r'[^a-z0-9\s]', '', teks)

    # Rapikan spasi
    teks = re.sub(r'\s+', ' ', teks).strip()

    # Ambil maksimal n kata pertama
    kata = teks.split()
    return ' '.join(kata[:n])

df.loc[mask_twitter, 'teks_bersih'] = df.loc[mask_twitter, 'teks_bersih'].apply(normalisasi_twitter)

print('✅ Lowercase & hapus tanda baca selesai')
print(f'\nContoh hasil (3 baris pertama):')
display(df.loc[mask_twitter, ['sumber', 'teks_bersih']].head(3))

✅ Lowercase & hapus tanda baca selesai

Contoh hasil (3 baris pertama):


,sumber,teks_bersih
10000,twitter,koh alung krisis air pemerintah berupaya menan...
10001,twitter,rizki fadillnaldo krisis air bersih jangan asa...
10002,twitter,indra kusuma air keruh mangkanya doi dkk kecew...


5.3. Hapus Baris Twitter yang Terlalu Pendek Setelah Cleaning

In [9]:
n_sebelum = len(df)

# Hapus baris yang teksnya < 15 karakter setelah dibersihkan
# (artinya isi aslinya hampir semua noise)
mask_terlalu_pendek = mask_twitter & (df['teks_bersih'].str.len() < 15)
df = df[~mask_terlalu_pendek].reset_index(drop=True)

n_drop = n_sebelum - len(df)
print(f'Drop baris twitter terlalu pendek: -{n_drop} baris')
print(f'Sisa total                        : {len(df)} baris')

Drop baris twitter terlalu pendek: -5 baris
Sisa total                        : 11540 baris


6. Verifikasi Akhir

In [10]:
mask_kaggle_baru  = df['sumber'] == 'berita_kaggle'
mask_twitter_baru = df['sumber'] == 'twitter'

# Cek sisa noise kaggle
print('--- Sisa noise berita_kaggle ---')
pola_cek_kaggle = ['baca berita dengan sedikit iklan', 'scroll ke bawah', 'klik di sini']
for p in pola_cek_kaggle:
    sisa = df.loc[mask_kaggle_baru, 'teks_bersih'].str.lower().str.contains(p, na=False).sum()
    status = '✅ Bersih' if sisa == 0 else f'⚠️  Masih ada {sisa} baris'
    print(f'  "{p}": {status}')

# Cek sisa noise twitter
print('\n--- Sisa noise twitter ---')
cek_devanagari = df.loc[mask_twitter_baru, 'teks_bersih'].str.contains(r'[\u0900-\u097F]', regex=True, na=False).sum()
cek_kapital    = df.loc[mask_twitter_baru, 'teks_bersih'].str.contains(r'[A-Z]', regex=True, na=False).sum()
cek_tanda_baca = df.loc[mask_twitter_baru, 'teks_bersih'].str.contains(r'[^a-z0-9\s]', regex=True, na=False).sum()
print(f'  Aksara Devanagari : {cek_devanagari} baris')
print(f'  Huruf kapital     : {cek_kapital} baris')
print(f'  Tanda baca        : {cek_tanda_baca} baris')

# Statistik panjang teks
print('\n--- Rata-rata jumlah kata per sumber ---')
print(df.groupby('sumber')['teks_bersih'].apply(lambda x: x.str.split().str.len().mean()).round(1))

--- Sisa noise berita_kaggle ---
  "baca berita dengan sedikit iklan": ✅ Bersih
  "scroll ke bawah": ✅ Bersih
  "klik di sini": ✅ Bersih

--- Sisa noise twitter ---
  Aksara Devanagari : 0 baris
  Huruf kapital     : 0 baris
  Tanda baca        : 0 baris

--- Rata-rata jumlah kata per sumber ---
sumber
berita_kaggle    50.0
twitter          20.6
Name: teks_bersih, dtype: float64


7. Pelabelan Sentimen (Regex Word Boundary)

In [11]:
# --- KAMUS SENTIMEN ---
kamus_sentimen = {
    'Positif': [
        'bagus', 'baik', 'berhasil', 'sukses', 'mantap', 'keren', 'hebat',
        'apresiasi', 'terima kasih', 'terimakasih', 'makasih', 'alhamdulillah',
        'syukur', 'senang', 'puas', 'lega', 'aman', 'selesai', 'tuntas',
        'meningkat', 'membaik', 'maju', 'berkembang', 'terpenuhi', 'tercapai',
        'optimal', 'responsif', 'cepat', 'tanggap', 'profesional', 'ramah', 'peduli'
    ],
    'Negatif': [
        'buruk', 'jelek', 'parah', 'kacau', 'hancur', 'rusak', 'gagal',
        'tidak ada', 'tidak layak', 'tidak mampu', 'marah', 'kesal', 'kecewa',
        'frustrasi', 'bosan', 'muak', 'jengkel', 'tidak puas', 'menyedihkan',
        'memprihatinkan', 'malu', 'miris', 'ironi', 'bohong', 'ingkar',
        'janji palsu', 'lambat', 'cuek', 'abai', 'semrawut', 'amburadul',
        'berantakan', 'kotor', 'bau', 'berbahaya', 'mematikan', 'susah',
        'sulit', 'rumit', 'ribet', 'dipersulit', 'pungli', 'korupsi', 'curang'
    ],
    'Netral': [
        'info', 'informasi', 'laporan', 'update', 'kabar', 'berita',
        'menurut', 'berdasarkan', 'data', 'fakta', 'kondisi', 'situasi',
        'terjadi', 'berlangsung', 'sedang', 'saat ini', 'diketahui',
        'disampaikan', 'diberitakan', 'dilaporkan', 'diumumkan',
        'pemerintah', 'dinas', 'instansi', 'rencana', 'program', 'kebijakan'
    ]
}

# Fungsi universal untuk menghitung kemunculan kata dengan batas kata (\b)
def hitung_skor_presisi(teks, daftar_keyword):
    skor = 0
    teks = str(teks)
    for kw in daftar_keyword:
        pola = r'\b' + re.escape(kw) + r'\b'
        if re.search(pola, teks):
            skor += 1
    return skor

# Fungsi penentuan sentimen
def label_sentimen_revisi(teks):
    skor_neg = hitung_skor_presisi(teks, kamus_sentimen['Negatif'])
    skor_pos = hitung_skor_presisi(teks, kamus_sentimen['Positif'])
    skor_net = hitung_skor_presisi(teks, kamus_sentimen['Netral'])
    
    # Aturan Konservatif: Utamakan deteksi Negatif jika terjadi seri
    if skor_neg >= skor_pos and skor_neg >= skor_net and skor_neg > 0:
        return 'Negatif'
    elif skor_pos >= skor_net and skor_pos > 0:
        return 'Positif'
    elif skor_net > 0:
        return 'Netral'
    else:
        return 'Netral' # Fallback jika tidak ada kata yang terdeteksi sama sekali

print("⏳ Sedang melabeli Sentimen pada teks bersih...")
df['sentimen'] = df['teks_bersih'].apply(label_sentimen_revisi)
print("✅ Pelabelan Sentimen selesai!")

⏳ Sedang melabeli Sentimen pada teks bersih...
✅ Pelabelan Sentimen selesai!


8. Pelabelan Urgensi (Regex Word Boundary)

In [12]:
# --- KAMUS URGENSI ---
kamus_urgensi = {
    'Tinggi': [
        'darurat', 'bahaya', 'kritis', 'gawat', 'parah', 'bencana', 'korban jiwa',
        'meninggal', 'tewas', 'kebakaran', 'banjir bandang', 'gempa', 'tsunami',
        'longsor', 'tenggelam', 'kecelakaan', 'luka berat', 'segera', 'mendesak',
        'urgent', 'sos', 'tolong', 'hancur', 'ambruk', 'roboh', 'evakuasi',
        'nyawa', 'koma', 'gawat darurat', 'igd', 'lumpuh', 'putus', 'terputus',
        'terendam', 'meluap', 'meledak', 'terbakar', 'hilang', 'kelaparan', 'krisis'
    ],
    'Sedang': [
        'rusak', 'bermasalah', 'gangguan', 'mati lampu', 'mati air', 'macet',
        'antrian', 'lambat', 'tidak berfungsi', 'bocor', 'kotor', 'bau',
        'berlubang', 'tidak layak', 'terganggu', 'perlu', 'harus', 'butuh',
        'mohon', 'minta', 'harap', 'keluhan', 'kurang', 'belum', 'tidak ada',
        'tidak merata', 'terhalang', 'terkendala', 'hambatan', 'sulit',
        'susah', 'kesulitan', 'terbatas', 'mahal', 'tidak terjangkau'
    ],
    'Rendah': [
        'saran', 'usul', 'masukan', 'semoga', 'harapan', 'mudah-mudahan',
        'diharapkan', 'agar', 'supaya', 'sebaiknya', 'seharusnya', 'mungkin',
        'cukup', 'lumayan', 'baik', 'bagus', 'terima kasih', 'apresiasi',
        'bagaimana', 'kapan', 'mengapa', 'kenapa', 'info', 'informasi',
        'tanya', 'update', 'kabar', 'perkembangan'
    ]
}

def label_urgensi_revisi(teks):
    skor_tinggi = hitung_skor_presisi(teks, kamus_urgensi['Tinggi'])
    skor_sedang = hitung_skor_presisi(teks, kamus_urgensi['Sedang'])
    skor_rendah = hitung_skor_presisi(teks, kamus_urgensi['Rendah'])
    
    # Pessimistic Priority: Jika seri, tarik ke tingkat bahaya yang lebih tinggi
    if skor_tinggi >= skor_sedang and skor_tinggi >= skor_rendah and skor_tinggi > 0:
        return 'Tinggi'
    elif skor_sedang >= skor_rendah and skor_sedang > 0:
        return 'Sedang'
    else:
        return 'Rendah' # Berlaku juga sebagai default jika skor semua 0

print("⏳ Sedang melabeli Urgensi pada teks bersih...")
df['urgensi'] = df['teks_bersih'].apply(label_urgensi_revisi)
print("✅ Pelabelan Urgensi selesai!")

⏳ Sedang melabeli Urgensi pada teks bersih...
✅ Pelabelan Urgensi selesai!


9. Pelabelan Ulang Kategori Isu (Multi-Label)

In [13]:
# --- KAMUS MULTI-LABEL ---
kamus_isu = {
    'kategori_bencana': [
        'gempa', 'tsunami', 'longsor', 'puting beliung', 'cuaca ekstrem',
        'hujan badai', 'erupsi', 'banjir bandang', 'kebakaran hutan',
        'karhutla', 'angin kencang', 'bencana alam', 'bnpb', 'evakuasi',
        'banjir susulan', 'tanggul jebol',
    ],
    'kategori_kesehatan': [
        'rumah sakit', 'puskesmas', 'bpjs', 'pasien', 'dokter', 'obat',
        'wabah', 'stunting', 'demam berdarah', 'covid', 'vaksin',
        'kesehatan', 'gizi buruk', 'nakes', 'tenaga kesehatan',
        'apotek', 'faskes', 'posyandu',
    ],
    'kategori_pendidikan': [
        'sekolah', 'guru', 'spp', 'zonasi', 'ppdb', 'kampus', 'mahasiswa',
        'beasiswa', 'kurikulum', 'ukt', 'siswa', 'pendidikan', 'ujian nasional',
        'sertifikasi guru', 'paud', 'literasi',
    ],
    'kategori_air_sanitasi': [
        'air bersih', 'pdam', 'krisis air', 'sanitasi', 'toilet', 'mck',
        'air keruh', 'kekeringan', 'air minum', 'sumur', 'irigasi',
        'kebocoran pipa', 'distribusi air',
    ],
    'kategori_transportasi': [
        'macet', 'krl', 'mrt', 'angkot', 'busway', 'transjakarta',
        'stasiun', 'terminal', 'kecelakaan lalu lintas', 'kemacetan',
        'ojek online', 'ojol', 'gojek', 'grab', 'lrt', 'bis kota',
        'angkutan umum', 'parkir liar', 'tilang',
    ],
    'kategori_keamanan': [
        'maling', 'begal', 'tawuran', 'kriminalitas', 'polisi', 'keamanan',
        'demo', 'konflik warga', 'pencurian', 'penipuan', 'narkoba',
        'kejahatan', 'kekerasan', 'aksi massa', 'perjudian',
    ],
    'kategori_pelayanan_publik': [
        'ktp', 'kk', 'dukcapil', 'paspor', 'birokrasi', 'pelayanan lambat',
        'pungli', 'sertifikat tanah', 'pelayanan publik', 'antrian panjang',
        'akta lahir', 'administrasi', 'kelurahan', 'kecamatan',
        'perizinan', 'izin usaha',
    ],
    'kategori_infrastruktur': [
        'jalan rusak', 'jembatan', 'aspal', 'berlubang', 'proyek jalan',
        'tol', 'gorong', 'trotoar', 'perbaikan jalan', 'jalan berlubang',
        'drainase', 'saluran air', 'infrastruktur', 'pembangunan jalan',
    ],
    'kategori_ekonomi': [
        'harga naik', 'sembako', 'inflasi', 'umkm', 'pasar', 'pedagang',
        'bantuan sosial', 'bansos', 'phk', 'pengangguran', 'upah minimum',
        'harga pangan', 'kemiskinan', 'kenaikan harga', 'bbm naik',
        'daya beli', 'ekonomi', 'investasi daerah',
    ],
    'kategori_lingkungan': [
        'sampah', 'tps', 'limbah', 'polusi', 'pencemaran', 'banjir',
        'sungai kotor', 'bau', 'berserakan', 'pengelolaan sampah',
        'tempat pembuangan', 'ruang hijau', 'penghijauan', 'illegal dumping',
    ],
}

def deteksi_kategori_presisi(teks, daftar_keyword):
    teks = str(teks)
    for kw in daftar_keyword:
        pola = r'\b' + re.escape(kw) + r'\b'
        if re.search(pola, teks):
            return 1 # Return 1 jika ada kata kunci yang cocok secara utuh
    return 0

# Timpa nilai kolom kategori lama dengan hasil scan teks yang sudah bersih
print("⏳ Sedang memindai ulang 10 Kategori Isu...")
for nama_kolom, kata_kunci in kamus_isu.items():
    df[nama_kolom] = df['teks_bersih'].apply(lambda x: deteksi_kategori_presisi(x, kata_kunci))

print("✅ Pelabelan Multi-Label selesai!")

⏳ Sedang memindai ulang 10 Kategori Isu...
✅ Pelabelan Multi-Label selesai!


7. Ringkasan & Ekspor Data Final

In [14]:
# Ekspor ke file baru agar data original tetap aman
path_output = '../data/processed/data_final_clean.csv'

try:
    df.to_csv(path_output, index=False, encoding='utf-8-sig')
    print(f'✅ Sukses! Data diekspor ke: {path_output}')
except Exception as e:
    print(f'❌ Gagal menyimpan file: {e}')

# Ringkasan akhir
print(f'\nShape output   : {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Kolom final    : {df.columns.tolist()}')
print(f'\nDistribusi sumber akhir:')
print(df['sumber'].value_counts())

print('\nPreview Data Final:')
display(df.head(3))

✅ Sukses! Data diekspor ke: ../data/processed/data_final_clean.csv

Shape output   : 11540 baris x 15 kolom
Kolom final    : ['sumber', 'tanggal', 'teks_bersih', 'urgensi', 'sentimen', 'kategori_infrastruktur', 'kategori_lingkungan', 'kategori_air_sanitasi', 'kategori_bencana', 'kategori_transportasi', 'kategori_pelayanan_publik', 'kategori_ekonomi', 'kategori_keamanan', 'kategori_pendidikan', 'kategori_kesehatan']

Distribusi sumber akhir:
sumber
berita_kaggle    10000
twitter           1540
Name: count, dtype: int64

Preview Data Final:


,sumber,tanggal,teks_bersih,urgensi,sentimen,kategori_infrastruktur,kategori_lingkungan,kategori_air_sanitasi,kategori_bencana,kategori_transportasi,kategori_pelayanan_publik,kategori_ekonomi,kategori_keamanan,kategori_pendidikan,kategori_kesehatan
0,berita_kaggle,2026-03-26,tempo co jakarta gubernur jakarta pramono anun...,Rendah,Positif,0,0,1,0,0,0,0,0,0,0
1,berita_kaggle,2026-03-26,dinas lingkungan hidup dlh provinsi dki jakart...,Rendah,Netral,0,0,0,0,0,0,0,0,0,0
2,berita_kaggle,2026-03-26,kompas com menteri dalam negeri mendagri muham...,Rendah,Netral,0,0,0,0,0,0,0,0,0,0


1. Audit Statistik Dataset (Identifikasi Masalah)

In [15]:
# Load dataset terakhir
path_input = '../data/processed/data_final_clean.csv'
df = pd.read_csv(path_input)

print("=== STATISTIK DATASET AWAL ===")
print(f"Total Baris Data: {len(df)}")

# 1. Cek Distribusi Urgensi (%)
print("\n1. Distribusi Urgensi:")
print(df['urgensi'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

# 2. Cek Distribusi Sentimen (%)
print("\n2. Distribusi Sentimen:")
print(df['sentimen'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

# 3. Hitung Data Tanpa Kategori
kategori_cols = [c for c in df.columns if c.startswith('kategori_')]

# Buat kolom bantuan untuk menghitung jumlah label per baris
df['jumlah_label'] = df[kategori_cols].sum(axis=1)

tanpa_label = len(df[df['jumlah_label'] == 0])
persentase_tanpa_label = (tanpa_label / len(df)) * 100

print(f"\n3. Masalah Kritis (Data Tanpa Kategori/Label):")
print(f"Jumlah: {tanpa_label} baris ({persentase_tanpa_label:.1f}% dari total data)")

=== STATISTIK DATASET AWAL ===
Total Baris Data: 11540

1. Distribusi Urgensi:
urgensi
Rendah    69.4%
Tinggi    16.3%
Sedang    14.4%
Name: proportion, dtype: object

2. Distribusi Sentimen:
sentimen
Netral     80.8%
Negatif    11.2%
Positif     8.0%
Name: proportion, dtype: object

3. Masalah Kritis (Data Tanpa Kategori/Label):
Jumlah: 5457 baris (47.3% dari total data)


2. Pembuangan Baris Tanpa Label (Drop Kosong)

In [16]:
print("⏳ Proses pembuangan data tanpa label sedang berjalan...")

# Filter hanya data yang memiliki minimal 1 label
df_valid = df[df['jumlah_label'] > 0].copy()

# Hapus kolom bantuan karena sudah tidak dibutuhkan
df_valid = df_valid.drop(columns=['jumlah_label'])

# Statistik setelah di-drop
print("\n=== HASIL SETELAH DI-DROP ===")
print(f"Total data awal    : {len(df)} baris")
print(f"Total data valid   : {len(df_valid)} baris")
print(f"Data yang terbuang : {len(df) - len(df_valid)} baris sampah")

print("\n=== DISTRIBUSI KATEGORI FINAL (Bisa >1 Label per Baris) ===")
print(df_valid[kategori_cols].sum().sort_values(ascending=False))

# Ekspor dataset yang sudah sangat valid untuk di-split
path_valid = '../data/final/data_valid_siap_split.csv'
df_valid.to_csv(path_valid, index=False)

print(f"\n✅ Selesai! Data valid berhasil diamankan di: {path_valid}")

⏳ Proses pembuangan data tanpa label sedang berjalan...

=== HASIL SETELAH DI-DROP ===
Total data awal    : 11540 baris
Total data valid   : 6083 baris
Data yang terbuang : 5457 baris sampah

=== DISTRIBUSI KATEGORI FINAL (Bisa >1 Label per Baris) ===
kategori_ekonomi             1101
kategori_lingkungan          1022
kategori_transportasi         832
kategori_pelayanan_publik     802
kategori_keamanan             802
kategori_infrastruktur        709
kategori_pendidikan           581
kategori_kesehatan            545
kategori_air_sanitasi         528
kategori_bencana              493
dtype: int64

✅ Selesai! Data valid berhasil diamankan di: ../data/final/data_valid_siap_split.csv


4. Pemisahan Data (Train & Test Split)

In [17]:
# 1. Load data yang sudah di-drop (dari cell sebelumnya)
path_valid = '../data/final/data_valid_siap_split.csv'
df_valid = pd.read_csv(path_valid)

print(f"Total Data Valid: {len(df_valid)} baris")

# 2. Definisikan Kolom Fitur (X) dan Target (Y)
kategori_cols = [c for c in df_valid.columns if c.startswith('kategori_')]
kolom_fitur = [c for c in df_valid.columns if c not in kategori_cols]

X = df_valid[kolom_fitur]  # Berisi teks_bersih, urgensi, sentimen, dll
Y = df_valid[kategori_cols] # Berisi 10 kolom target klasifikasi

# 3. Inisialisasi Splitting (80% Train, 20% Test)
# n_splits=1 karena kita hanya membagi satu kali (bukan Cross Validation)
msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# 4. Eksekusi Splitting
for train_index, test_index in msss.split(X, Y):
    # Buat dataframe utuh untuk masing-masing set menggunakan indeks hasil split
    df_train = df_valid.iloc[train_index].copy()
    df_test = df_valid.iloc[test_index].copy()

# 5. Verifikasi Distribusi Hasil Split
print("\n=== HASIL PEMBAGIAN DATA ===")
print(f"Train Set : {len(df_train)} baris ({len(df_train)/len(df_valid)*100:.0f}%)")
print(f"Test Set  : {len(df_test)} baris ({len(df_test)/len(df_valid)*100:.0f}%)")

print("\n=== CEK KESEIMBANGAN PROPORSIONAL (Contoh 3 Kategori Teratas) ===")
# Kita cek apakah pembagiannya adil (sekitar 80:20 untuk setiap label)
top_3_kategori = Y.sum().sort_values(ascending=False).head(3).index

for kat in top_3_kategori:
    total = df_valid[kat].sum()
    di_train = df_train[kat].sum()
    di_test = df_test[kat].sum()
    print(f"{kat.replace('kategori_', '').title()}:")
    print(f"  Total: {total} -> Train: {di_train} ({(di_train/total)*100:.1f}%), Test: {di_test} ({(di_test/total)*100:.1f}%)")

# 6. Ekspor Data Final Siap Modeling
path_train = '../data/final/train_data.csv'
path_test = '../data/final/test_data.csv'

df_train.to_csv(path_train, index=False)
df_test.to_csv(path_test, index=False)

print(f"\n✅ File Train dan Test berhasil disimpan secara terpisah!")

Total Data Valid: 6083 baris

=== HASIL PEMBAGIAN DATA ===
Train Set : 4860 baris (80%)
Test Set  : 1223 baris (20%)

=== CEK KESEIMBANGAN PROPORSIONAL (Contoh 3 Kategori Teratas) ===
Ekonomi:
  Total: 1101 -> Train: 881 (80.0%), Test: 220 (20.0%)
Lingkungan:
  Total: 1022 -> Train: 818 (80.0%), Test: 204 (20.0%)
Transportasi:
  Total: 832 -> Train: 666 (80.0%), Test: 166 (20.0%)

✅ File Train dan Test berhasil disimpan secara terpisah!


5. Label Encoding (Sentimen & Urgensi)

In [18]:
# Load data yang sudah di-split sebelumnya
df_train = pd.read_csv('../data/final/train_data.csv')
df_test = pd.read_csv('../data/final/test_data.csv')

# Definisikan kamus pemetaan (Mapping)
# Diurutkan dari tingkat paling rendah/negatif ke tinggi/positif
map_urgensi = {'Rendah': 0, 'Sedang': 1, 'Tinggi': 2}
map_sentimen = {'Negatif': 0, 'Netral': 1, 'Positif': 2}

print("⏳ Melakukan Label Encoding...")

# Aplikasikan ke Data Train
df_train['urgensi'] = df_train['urgensi'].map(map_urgensi)
df_train['sentimen'] = df_train['sentimen'].map(map_sentimen)

# Aplikasikan ke Data Test
df_test['urgensi'] = df_test['urgensi'].map(map_urgensi)
df_test['sentimen'] = df_test['sentimen'].map(map_sentimen)

print("✅ Label Encoding Selesai! Target sekarang berupa angka (0, 1, 2).")

⏳ Melakukan Label Encoding...
✅ Label Encoding Selesai! Target sekarang berupa angka (0, 1, 2).


6. Undersampling (Khusus Data Latih)

In [19]:
print("📊 Distribusi Urgensi Sebelum Undersampling:")
print(df_train['urgensi'].value_counts())

# Pisahkan data berdasarkan kelas urgensi
df_urgensi_0 = df_train[df_train['urgensi'] == 0] # Rendah (Mayoritas)
df_urgensi_1 = df_train[df_train['urgensi'] == 1] # Sedang
df_urgensi_2 = df_train[df_train['urgensi'] == 2] # Tinggi

# Cari batas jumlah data dari kelas yang paling sedikit
batas_minoritas = min(len(df_urgensi_0), len(df_urgensi_1), len(df_urgensi_2))

# Potong kelas yang lebih besar agar sama dengan batas minoritas
df_urgensi_0_down = resample(df_urgensi_0, replace=False, n_samples=batas_minoritas, random_state=42)
df_urgensi_1_down = resample(df_urgensi_1, replace=False, n_samples=batas_minoritas, random_state=42)
# Urgensi 2 di-resample untuk konsistensi kode
df_urgensi_2_down = resample(df_urgensi_2, replace=False, n_samples=batas_minoritas, random_state=42)

# Gabungkan kembali dan acak urutan barisnya (shuffle)
df_train_balanced = pd.concat([df_urgensi_0_down, df_urgensi_1_down, df_urgensi_2_down])
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Simpan hasil undersampling ke file baru agar data original tetap aman
path_train_baru = '../data/final/train_data_balanced.csv'
path_test_baru = '../data/final/test_data_encoded.csv'

df_train_balanced.to_csv(path_train_baru, index=False)
df_test.to_csv(path_test_baru, index=False)

print(f"\n Undersampling Selesai! Data diseimbangkan di angka {batas_minoritas} per kelas.")
print(f"File latih khusus AI disimpan di: {path_train_baru}")
print(f"File uji khusus AI disimpan di: {path_test_baru}")

📊 Distribusi Urgensi Sebelum Undersampling:
urgensi
0    2959
2    1088
1     813
Name: count, dtype: int64

 Undersampling Selesai! Data diseimbangkan di angka 813 per kelas.
File latih khusus AI disimpan di: ../data/final/train_data_balanced.csv
File uji khusus AI disimpan di: ../data/final/test_data_encoded.csv


7. Verifikasi Proporsi Akhir (Data Latih Seimbang)

In [20]:
print("=== DISTRIBUSI DATA LATIH (TRAIN) SETELAH UNDERSAMPLING ===\n")
print(f"Total Baris Data Latih Baru: {len(df_train_balanced)}\n")

print("[ 1. Distribusi Urgensi ]")
print(df_train_balanced['urgensi'].value_counts().sort_index())

print("\n[ 2. Distribusi Sentimen ]")
print(df_train_balanced['sentimen'].value_counts().sort_index())

print("\n[ 3. Distribusi Kategori Isu (Multi-label) ]")
kat_cols = [c for c in df_train_balanced.columns if c.startswith('kategori_')]
print(df_train_balanced[kat_cols].sum().sort_values(ascending=False))

=== DISTRIBUSI DATA LATIH (TRAIN) SETELAH UNDERSAMPLING ===

Total Baris Data Latih Baru: 2439

[ 1. Distribusi Urgensi ]
urgensi
0    813
1    813
2    813
Name: count, dtype: int64

[ 2. Distribusi Sentimen ]
sentimen
0     433
1    1839
2     167
Name: count, dtype: int64

[ 3. Distribusi Kategori Isu (Multi-label) ]
kategori_lingkungan          410
kategori_ekonomi             398
kategori_transportasi        369
kategori_infrastruktur       348
kategori_pelayanan_publik    300
kategori_bencana             270
kategori_keamanan            255
kategori_pendidikan          224
kategori_air_sanitasi        209
kategori_kesehatan           208
dtype: int64
